# Using GridSearchCV to find best model and perform hyper parameter tuning: Iris dataset

In [1]:
import pandas as pd

df = pd.read_csv("data_iris.csv")
print(df.sample(10))

     sepal_length  sepal_width  petal_length  petal_width     species
129           7.2          3.0           5.8          1.6   virginica
72            6.3          2.5           4.9          1.5  versicolor
34            4.9          3.1           1.5          0.1      setosa
42            4.4          3.2           1.3          0.2      setosa
53            5.5          2.3           4.0          1.3  versicolor
91            6.1          3.0           4.6          1.4  versicolor
116           6.5          3.0           5.5          1.8   virginica
13            4.3          3.0           1.1          0.1      setosa
56            6.3          3.3           4.7          1.6  versicolor
41            4.5          2.3           1.3          0.3      setosa


In [2]:
# Count rows and columns

print(df.shape)

(150, 5)


## Approach 1 (BAD): Use train_test_split and manually tune parameters by trial and error

In [3]:
# step1: Separate features(X) and labels(y):
X = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = df["species"]

In [4]:
# step2) Print X and y
print("X:\n", X)
print("y:\n", y)

X:
      sepal_length  sepal_width  petal_length  petal_width
0             5.1          3.5           1.4          0.2
1             4.9          3.0           1.4          0.2
2             4.7          3.2           1.3          0.2
3             4.6          3.1           1.5          0.2
4             5.0          3.6           1.4          0.2
..            ...          ...           ...          ...
145           6.7          3.0           5.2          2.3
146           6.3          2.5           5.0          1.9
147           6.5          3.0           5.2          2.0
148           6.2          3.4           5.4          2.3
149           5.9          3.0           5.1          1.8

[150 rows x 4 columns]
y:
 0         setosa
1         setosa
2         setosa
3         setosa
4         setosa
         ...    
145    virginica
146    virginica
147    virginica
148    virginica
149    virginica
Name: species, Length: 150, dtype: object


In [5]:
# step3: Split the data into training and testing sets

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True)

In [6]:
print("Shape of training dataset:", X_train.shape) # 105 rows and 4 columns
print("Shape of testing dataset:", X_test.shape) # 105 rows and 4 columns

Shape of training dataset: (105, 4)
Shape of testing dataset: (45, 4)


In [7]:
from sklearn.svm import SVC

model = SVC(kernel='linear', # linear’, ‘poly’, ‘rbf’, ‘sigmoid’,
            C=50,            # C = 10, 20, ..., 50
            gamma='auto',    # auto, scale
            degree = 4,
            decision_function_shape = 'ovr') # Number of combination = 4 kernels x 5 C x 2 gammas = 40
model.fit(X_train,y_train)
model.score(X_test, y_test)

0.9555555555555556

### How many combinations of hyperparameters ?
- **4 x 5 x 2 = 40**
- This would take a long time if done manually

## Approach 2: Use K Fold Cross validation with different hyperparamters manually

**Manually try suppling models with different parameters to cross_val_score function with 5 fold cross validation**

In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
import numpy as np

In [9]:
score = cross_val_score(SVC(kernel='linear',C=1, gamma='auto'), X, y, cv=5)

print(score)
print(np.mean(score))

[0.96666667 1.         0.96666667 0.96666667 1.        ]
0.9800000000000001


In [10]:
score = cross_val_score(SVC(kernel='rbf',  C=10, gamma='auto'), X, y, cv=5) # C=1,10

print(score)
print(np.mean(score))

[0.96666667 1.         0.96666667 0.96666667 1.        ]
0.9800000000000001


In [11]:
score = cross_val_score(SVC(kernel='rbf', C=20, gamma='auto'), X, y, cv=5)

print(score)
print(np.mean(score))

[0.96666667 1.         0.9        0.96666667 1.        ]
0.9666666666666668


### This will take a long time
**Based on few HP, from above results we can say that kernel=rbf with C=10 will give best performance**

## Approach 3: Use GridSearchCV
**GridSearchCV does exactly same thing as above but in a single line of code**

In [17]:
from sklearn.model_selection import GridSearchCV

model_gs = GridSearchCV(SVC(gamma='auto'), {
    'C': [1,10,20],
    'kernel': ['rbf','linear']
}, cv=5, return_train_score=False)

model_gs.fit(X, y)
print(model_gs.cv_results_)

{'mean_fit_time': array([0.00360432, 0.00371652, 0.00619721, 0.00300269, 0.00249949,
       0.0026041 ]), 'std_fit_time': array([0.00079673, 0.00097956, 0.00278077, 0.0006413 , 0.00063835,
       0.00048068]), 'mean_score_time': array([0.0024044 , 0.00299921, 0.00299945, 0.00200253, 0.00179472,
       0.00159225]), 'std_score_time': array([4.87706271e-04, 8.84410698e-04, 8.94030451e-04, 5.41666863e-06,
       3.97280924e-04, 4.99352255e-04]), 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20],
             mask=[False, False, False, False, False, False],
       fill_value='?',
            dtype=object), 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
             mask=[False, False, False, False, False, False],
       fill_value='?',
            dtype=object), 'params': [{'C': 1, 'kernel': 'rbf'}, {'C': 1, 'kernel': 'linear'}, {'C': 10, 'kernel': 'rbf'}, {'C': 10, 'kernel': 'linear'}, {'C': 20, 'kernel': 'rbf'}, {'C': 20, 'kernel': 'linear'}], 's

In [18]:
df = pd.DataFrame(model_gs.cv_results_)
print(df)

   mean_fit_time  std_fit_time  mean_score_time  std_score_time param_C  \
0       0.003604      0.000797         0.002404        0.000488       1   
1       0.003717      0.000980         0.002999        0.000884       1   
2       0.006197      0.002781         0.002999        0.000894      10   
3       0.003003      0.000641         0.002003        0.000005      10   
4       0.002499      0.000638         0.001795        0.000397      20   
5       0.002604      0.000481         0.001592        0.000499      20   

  param_kernel                         params  split0_test_score  \
0          rbf      {'C': 1, 'kernel': 'rbf'}           0.966667   
1       linear   {'C': 1, 'kernel': 'linear'}           0.966667   
2          rbf     {'C': 10, 'kernel': 'rbf'}           0.966667   
3       linear  {'C': 10, 'kernel': 'linear'}           1.000000   
4          rbf     {'C': 20, 'kernel': 'rbf'}           0.966667   
5       linear  {'C': 20, 'kernel': 'linear'}           1.000000  

In [19]:
df[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,1,linear,0.980000
2,10,rbf,0.980000
3,10,linear,0.973333
4,20,rbf,0.966667
5,20,linear,0.966667


In [20]:
model_gs.best_params_

{'C': 1, 'kernel': 'rbf'}

In [21]:
model_gs.best_score_

0.9800000000000001

## Now lets use GridsearchCV on different models with different hyperparameters

In [22]:
# step1) 
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

model_params = {
    'svm': {
        'model': SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }  
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params' : {
            'n_estimators': [1,5,10]
        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(solver='liblinear', multi_class='auto'),
        'params': {
            'C': [1,5,10]
        }
    }
}


In [23]:
# step2) print each models best score and best parameter
scores = []

for model_name, mp in model_params.items():
    model_gs =  GridSearchCV(mp['model'], mp['params'], cv=5, return_train_score=False)
    model_gs.fit(X, y)
    scores.append({
        'model': model_name,
        'best_score': model_gs.best_score_,
        'best_params': model_gs.best_params_
    })
    
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
print(df)

                 model  best_score                best_params
0                  svm    0.980000  {'C': 1, 'kernel': 'rbf'}
1        random_forest    0.966667       {'n_estimators': 10}
2  logistic_regression    0.966667                   {'C': 5}


### Conclusion: SVM with C=1 and kernel='rbf' is the best model

## Approach4: Use RandomizedSearchCV
- **Use RandomizedSearchCV to reduce number of iterations** with random combination of parameters. This is useful when you have too many parameters to try and your training time is longer. 
- It helps reduce the cost of computation
- Does not guarantee the best model


In [24]:
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV

model_rs = RandomizedSearchCV(SVC(gamma='auto'), {
        'C': [1,10,20],
        'kernel': ['rbf','linear']
    }, 
    cv=5, 
    return_train_score=False, 
    n_iter=2
)
model_rs.fit(X, y)
pd.DataFrame(model_rs.cv_results_)[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,20,linear,0.966667


### Conclusion: Best SVC model hyperparameters are C=1. 
- The result may be different than that of GridSearchCV because it uses randomized search